In [3]:

import sys
import json
from pathlib import Path
from tqdm import tqdm

import spacy
import pandas as pd


In [9]:
# DATA DIRECTORIES
PROJECT_ROOT = Path.cwd().parent.parent
DATA_DIR = PROJECT_ROOT / 'data' / 'newyorker_caption_contest'
JSON_DIR = DATA_DIR / 'contests.json'
CSV_CLEAN_DIR = DATA_DIR / 'cleaned_data'





## 0. Cleaning the data set with spacy

We used spacy to clean the raw data set to have more meaningfull themes

In [10]:

# docs = []

# for csv_path in sorted(CSV_CLEAN_DIR.glob("*.csv")):
#     df = pd.read_csv(csv_path)
#     contest_id = csv_path.stem

#     for idx, row in df.iterrows():
#         text = str(row.get("caption", "")).strip()
#         if not text:
#             continue

#         docs.append({
#             "contest_id": contest_id,
#             "caption_id": idx,
#             "text": text,

#             # metadata (if present)
#             "rank": row.get("rank"),
#             "mean": row.get("mean"),
#             "precision": row.get("precision"),
#             "votes": row.get("votes"),
#             "funny": row.get("funny"),
#             "somewhat_funny": row.get("somewhat_funny"),
#             "not_funny": row.get("not_funny"),
#         })

# docs_df = pd.DataFrame(docs)
# print("docs_df shape:", docs_df.shape)


In [11]:
def load_contests(contests_json_path: Path):
    with open(contests_json_path, "r", encoding="utf-8") as f:
        contests = json.load(f)
    return contests

def build_docs_dataframe(debug=False):
    contests = load_contests(JSON_DIR)
    docs = []
    
    failed_csv = []

    # tqdm over contests
    for contest in tqdm(contests, desc="Step 1: data loading/building"):
        cid = contest["contest_id"]
        
        # print DEBUG
        # print(f"Processing contest ID: {cid}")

        raw_meta = contest.get("metadata")
        meta = raw_meta or {}   # normalize None -> {}

        # ---- Build one aggregated "image document" per contest ----
        parts = []

        for key in ["image_descriptions", "image_uncanny_descriptions",
                    "image_locations", "questions"]:
            values = meta.get(key) or []
            parts.extend(values)

        entities = meta.get("entities") or []
        for url in entities:
            last = str(url).rstrip("/").split("/")[-1]
            parts.append(last.replace("_", " "))

        image_text = " ".join(map(str, parts)).strip()

        if image_text:
            docs.append(
                {
                    "doc_id": f"image_{cid}",
                    "contest_id": cid,
                    "kind": "image",
                    "text": image_text,
                }
            )

        # ---- Add caption documents for this contest ----
        csv_path = CSV_CLEAN_DIR / f"{cid}.csv"
        if not csv_path.exists():
            csv_path = CSV_CLEAN_DIR / f"{cid}_cleaned.csv"
            if not csv_path.exists():
                print(f"Warning: CSV for contest {cid} not found at {CSV_CLEAN_DIR / f'{cid}.csv'} and neither at {csv_path}.")
                continue

        try:
            captions_df = pd.read_csv(csv_path)
        except Exception as e:
            print(f"Error reading CSV for contest {cid} at {csv_path}: {e}")
            failed_csv.append(cid)
            continue

        # optional inner tqdm if you want per-caption progress too
        for idx, row in captions_df.iterrows():
            cap_text = str(row.get("caption", "")).strip()
            if not cap_text:
                continue

            docs.append(
                {
                    "doc_id": f"caption_{cid}_{idx}",
                    "contest_id": cid,
                    "kind": "caption",
                    "text": cap_text,
                    "rank": row.get("rank", None),
                    "mean": row.get("mean", None),
                    "precision": row.get("precision", None),
                    "votes": row.get("votes", None),
                    "not_funny": row.get("not_funny", None),
                    "somewhat_funny": row.get("somewhat_funny", None),
                    "funny": row.get("funny", None),
                }
            )

    df = pd.DataFrame(docs)
    
    # keep only expected first 1000 rows for debugging
    if debug:
        df = df.head(5000)
    
    return df, failed_csv

In [29]:
docs_df, error_csv = build_docs_dataframe(debug=False)

docs_df.head()

Step 1: data loading/building: 100%|██████████| 384/384 [00:35<00:00, 10.94it/s]


,doc_id,contest_id,kind,text,rank,mean,precision,votes,not_funny,somewhat_funny,funny
0,image_510,510,image,A man is relaxing on a city street. Others are...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,caption_510_0,510,caption,I'm a congressman--obstruction is my job.,0.0,1.913043,0.094022,69.0,24.0,27.0,18.0
2,caption_510_1,510,caption,"I'm what they mean when they say, 'The middle ...",1.0,1.842105,0.191381,19.0,8.0,6.0,5.0
3,caption_510_2,510,caption,Does this suit make me look flat?,2.0,1.711111,0.112915,45.0,21.0,16.0,8.0
4,caption_510_3,510,caption,"When the right woman comes along, I'll know it.",3.0,1.625000,0.116657,32.0,15.0,14.0,3.0


In [30]:
# count the number of rows per kind
docs_df['kind'].value_counts()

# keep only kind 'image'
image_desc_df = docs_df.copy()
image_desc_df = image_desc_df[image_desc_df['kind'] == 'image']

# keep only relevant columns
image_desc_df = image_desc_df[['doc_id', 'contest_id', 'text']]
image_desc_df.head()

,doc_id,contest_id,text
0,image_510,510,A man is relaxing on a city street. Others are...
3906,image_511,511,A man in a winter coat and cap is looking at a...
7232,image_512,512,A man and woman are standing facing one anothe...
11632,image_513,513,Three business men are walking down a hall. Th...
15774,image_514,514,Some cowboys are riding through the desert. Th...


In [31]:
# Charger spaCy
nlp = spacy.load("en_core_web_sm")

# Activer tqdm pour pandas
tqdm.pandas()

def clean_caption(text):
    """
    Clean a caption by:
    - running spaCy
    - removing stop words and non-content words
    - keeping only alphabetic NOUN/PROPN/ADJ/VERB
    - lemmatizing and lowercasing
    """
    doc = nlp(str(text))  # str() in case of NaN
    tokens = []
    for token in doc:
        if (
            token.is_alpha
            and not token.is_stop
            and token.pos_ in {"NOUN", "PROPN", "ADJ", "VERB"}
        ):
            tokens.append(token.lemma_.lower())
    return " ".join(tokens)

# Apply with progress bar
image_desc_df["clean_text"] = image_desc_df["text"].progress_apply(clean_caption)


100%|██████████| 240/240 [00:04<00:00, 57.05it/s]


In [32]:
image_desc_df.head()

,doc_id,contest_id,text,clean_text
0,image_510,510,A man is relaxing on a city street. Others are...,man relax city street go business step pass ma...
3906,image_511,511,A man in a winter coat and cap is looking at a...,man winter coat cap look small bearded man hol...
7232,image_512,512,A man and woman are standing facing one anothe...,man woman stand face mirror yoga studio man lo...
11632,image_513,513,Three business men are walking down a hall. Th...,business man walk hall carry briefcase man wal...
15774,image_514,514,Some cowboys are riding through the desert. Th...,cowboy rid desert rock horse rock horse place ...


In [33]:
# Add clean_text column of the images to corresponding docs_df rows.
#  Example: image_desc_df has doc_id:image_510 => docs_df: row image_510 fill clean_text
for idx, row in image_desc_df.iterrows():
    doc_id = row['doc_id']
    clean_text = row['clean_text']
    docs_df.loc[docs_df['doc_id'] == doc_id, 'clean_text'] = clean_text

docs_df.head()

,doc_id,contest_id,kind,text,rank,mean,precision,votes,not_funny,somewhat_funny,funny,clean_text
0,image_510,510,image,A man is relaxing on a city street. Others are...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,man relax city street go business step pass ma...
1,caption_510_0,510,caption,I'm a congressman--obstruction is my job.,0.0,1.913043,0.094022,69.0,24.0,27.0,18.0,NaN
2,caption_510_1,510,caption,"I'm what they mean when they say, 'The middle ...",1.0,1.842105,0.191381,19.0,8.0,6.0,5.0,NaN
3,caption_510_2,510,caption,Does this suit make me look flat?,2.0,1.711111,0.112915,45.0,21.0,16.0,8.0,NaN
4,caption_510_3,510,caption,"When the right woman comes along, I'll know it.",3.0,1.625000,0.116657,32.0,15.0,14.0,3.0,NaN


In [42]:
print("Total documents:", len(docs_df))
print("\n",docs_df["kind"].value_counts())
print("\n",docs_df["clean_text"].notnull().value_counts())

Total documents: 2263328

 kind
caption    2263088
image          240
Name: count, dtype: int64

 clean_text
False    2263088
True         240
Name: count, dtype: int64


In [82]:
caption_clean_df = pd.read_csv("/Users/jeremyserillon/Desktop/ada-2025-project-json-derulo/data/newyorker_caption_contest/spacy/spacy_captions.csv")

# strip either suffix "_cleaned" or prefix "cleaned_"
clean_contest_id = (
    caption_clean_df["contest_id"].astype(str)
    .str.replace(r"^cleaned_", "", regex=True)
    .str.replace(r"_cleaned$", "", regex=True)
)

# build the caption/doc id
caption_clean_df = caption_clean_df.assign(
    doc_id="caption_" + clean_contest_id + "_" + caption_clean_df["caption_id"].astype(str)
).set_index("doc_id").drop(columns=["contest_id", "rank", "mean", "precision", "votes", "not_funny", "somewhat_funny", "funny"])

print(caption_clean_df.head())

               caption_id                                               text  \
doc_id                                                                         
caption_510_0           0          I'm a congressman--obstruction is my job.   
caption_510_1           1  I'm what they mean when they say, 'The middle ...   
caption_510_2           2                  Does this suit make me look flat?   
caption_510_3           3    When the right woman comes along, I'll know it.   
caption_510_4           4  I used to lie in the gutter, but then I quit d...   

                                clean_text  
doc_id                                      
caption_510_0  congressman obstruction job  
caption_510_1       mean middle class step  
caption_510_2               suit look flat  
caption_510_3        right woman come know  
caption_510_4     lie gutter quit drinking  


In [85]:
# Fill the clean_text column of docs_df with the corresponding from caption_clean_df's (by matching doc_id)

# caption_clean_df is indexed by doc_id
lookup = caption_clean_df["clean_text"]  # index = doc_id

docs_df["clean_text"] = docs_df["clean_text"].combine_first(
    docs_df["doc_id"].map(lookup)
)

docs_df.head()

,doc_id,contest_id,kind,text,rank,mean,precision,votes,not_funny,somewhat_funny,funny,clean_text
0,image_510,510,image,A man is relaxing on a city street. Others are...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,man relax city street go business step pass ma...
1,caption_510_0,510,caption,I'm a congressman--obstruction is my job.,0.0,1.913043,0.094022,69.0,24.0,27.0,18.0,congressman obstruction job
2,caption_510_1,510,caption,"I'm what they mean when they say, 'The middle ...",1.0,1.842105,0.191381,19.0,8.0,6.0,5.0,mean middle class step
3,caption_510_2,510,caption,Does this suit make me look flat?,2.0,1.711111,0.112915,45.0,21.0,16.0,8.0,suit look flat
4,caption_510_3,510,caption,"When the right woman comes along, I'll know it.",3.0,1.625000,0.116657,32.0,15.0,14.0,3.0,right woman come know


In [ ]:
cols_export = ["doc_id", "contest_id", "kind", "text", "clean_text", "rank", "mean",
               "precision", "votes", "not_funny", "somewhat_funny", "funny"]
# export csv
docs_df.to_csv("/Users/jeremyserillon/Downloads/docs_clean_text.csv", columns=cols_export, index=False)

In [91]:
# count empty clean_text
docs_df["clean_text"].isna().sum()

np.int64(15663)